In [1]:
import jax
import jax.numpy as jnp
!pip install -U flax
import flax
import flax.linen as nn
import optax
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.9/533.9 kB 6.4 MB/s  0:00:00
  Attempting uninstall: flax
    Found existing installation: flax 0.11.2
    Uninstalling flax-0.11.2:
      Successfully uninstalled flax-0.11.2


In [2]:
import jax, flax
print("jax:", jax.__version__, "flax:", flax.__version__)

jax: 0.11.1 flax: 0.12.12


## MLP projector (input_dim -> final_dim through projection_dim)

In [3]:
# Project representations from "initial dimension" to "z_dim" through a single layer "projector_dim"
class Projector(nn.Module):
    projector_dim: int
    z_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.projector_dim)(x)
        x = nn.silu(x)

        x = nn.Dense(self.projector_dim)(x)
        x = nn.silu(x)

        x = nn.Dense(self.z_dim)(x)

        return x

## Modulation (shift, scale)

In [4]:
def modulate(x, shift, scale):
  """
  x:     [B, num_patches, D]
  scale: [B, D]
  shift: [B, D]
  """
  return x * (1.0 + scale[:, None, :]) + shift[:, None, :]

## Time and class embedding

In [5]:
# Embed timestep using fourier positional emcoding scheme and project that into the required "hidden_size"

class TimestepEmbedder(nn.Module):
    hidden_size: int # Final dim of encoded timestep
    frequency_embedding_size: int = 256 # fourier encoding embedding dimension (number of frequencies * 2)

    @staticmethod
    def positional_embedding(t, dim, max_period=10000):
        """
        t : [B, 1]
        dim : int
        max_period : int
        """
        half = dim // 2

        freqs = jnp.exp(
            -jnp.log(max_period) *
            jnp.arange(half, dtype=jnp.float32) / half
        ) # freq = exp(-log(10000) * i/num_frequencies) for all i from 0 to num_frequencies - 1 [num_frequencies]

        args = t[:, None].astype(jnp.float32) * freqs[None, :] # t[:, None] : [B, 1] freq[None, :] : [1, num_frequencies] args : [B, num_frequencies]

        embedding = jnp.concatenate(
            [jnp.cos(args), jnp.sin(args)],
            axis=-1
        ) # [cos(tw0), cos(tw1) .. cos(tw127), sin(tw0), sin(tw1), ... sin(tw127)] [B, frequency_embedding_size]

        if dim % 2:
            embedding = jnp.concatenate(
                [embedding, jnp.zeros_like(embedding[:, :1])],
                axis=-1
            )

        return embedding

    @nn.compact
    def __call__(self, t):

        t_freq = self.positional_embedding(
            t, self.frequency_embedding_size
        )
        # project from dim: frequency_embedding_size to hidden_size
        x = nn.Dense(self.hidden_size)(t_freq)
        x = nn.silu(x)
        x = nn.Dense(self.hidden_size)(x)

        return x

In [6]:
class LabelEmbedder(nn.Module):
    """
    Converts integer class labels into learnable embedding vectors.
    Supports label dropout for classifier-free guidance (CFG).
    """

    num_classes: int
    hidden_size: int
    dropout_prob: float

    @nn.compact
    def __call__(self, labels, train, force_drop_ids=None):

        # Learnable embedding lookup table:
        # [num_classes + 1, hidden_size]
        # Extra row represents the null/unconditional class.
        embedding_table = nn.Embed(
            num_embeddings=self.num_classes +
                           int(self.dropout_prob > 0),
            features=self.hidden_size
        ) # nn.Embed essentially creates a lookup table for each class to a corresponding encoded vector of dim : hidden_size (learnable!)

        # Randomly replace some labels with the null class during training.
        if train and self.dropout_prob > 0:

            # [B] Boolean mask: True means drop the class label.
            drop_ids = jax.random.bernoulli(
                self.make_rng("dropout"),
                self.dropout_prob,
                labels.shape
            )

            # Replace dropped labels with the null-class index.
            # labels: [B]
            labels = jnp.where(
                drop_ids,
                self.num_classes,
                labels
            )

        # Optionally force specific samples to use the null class (CFG).
        if force_drop_ids is not None:

            # force_drop_ids: [B]
            labels = jnp.where(
                force_drop_ids == 1,
                self.num_classes,
                labels
            )

        # Lookup corresponding embedding vectors.
        # labels: [B]
        # Output: [B, hidden_size]
        return embedding_table(labels)


## SiT block

In [7]:
class SiTBlock(nn.Module):
    """
    AdaLN-Zero Transformer block:
    x' = x + gate_msa * Attention(AdaLN(x))
    x'' = x' + gate_mlp * MLP(AdaLN(x'))

    The conditioning vector c = t_embed + y_embed combines the timestep
    embedding and class-label embedding. An MLP generates six vectors
    (shift, scale, gate for attention and MLP), shared across all tokens.
    """

    hidden_size: int
    num_heads: int
    mlp_ratio: float = 4.0
    qk_norm: bool = False

    @nn.compact
    def __call__(self, x, c):

        # x: [B, T, D] — token sequence (patch tokens + CLS token)
        # c: [B, D]    — conditioning = timestep embedding + class embedding

        # Attention normalization: LayerNorm without learnable affine parameters
        h = nn.LayerNorm(
            epsilon=1e-6,
            use_scale=False,
            use_bias=False
        )(x)  # [B, T, D]

        # Generate six conditioning vectors from c through SiLU + Dense.
        # modulation: [B, 6D]
        modulation = nn.Dense(
            6 * self.hidden_size,
            kernel_init=nn.initializers.zeros,
            bias_init=nn.initializers.zeros
        )(nn.silu(c))

        # Each vector: [B, D]
        shift_msa, scale_msa, gate_msa, \
        shift_mlp, scale_mlp, gate_mlp = jnp.split(
            modulation, 6, axis=-1
        )

        # Attention branch
        # AdaLN: h * (1 + scale_msa) + shift_msa
        # scale/shift: [B, D], broadcast across T tokens
        h = modulate(h, shift_msa, scale_msa)  # [B, T, D]

        h = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            qkv_features=self.hidden_size,
            out_features=self.hidden_size,
            use_bias=True,
            dropout_rate=0.0
        )(h)  # [B, T, D]

        # Gated residual connection; gate_msa: [B, D]
        x = x + gate_msa[:, None, :] * h  # [B, T, D]

        # MLP branch
        h = nn.LayerNorm(
            epsilon=1e-6,
            use_scale=False,
            use_bias=False
        )(x)  # [B, T, D]

        # AdaLN using MLP-specific scale and shift
        h = modulate(h, shift_mlp, scale_mlp)  # [B, T, D]

        # Two-layer feed-forward network
        h = nn.Dense(
            int(self.hidden_size * self.mlp_ratio)
        )(h)  # [B, T, 4D]

        h = nn.gelu(h, approximate=True)  # [B, T, 4D]

        h = nn.Dense(self.hidden_size)(h)  # [B, T, D]

        # Gated MLP residual connection; gate_mlp: [B, D]
        x = x + gate_mlp[:, None, :] * h  # [B, T, D]

        return x  # [B, T, D]

## Unified final layer (SiT, REPA, REG)

In [8]:
class FinalLayer(nn.Module):
    """
    Configurable final prediction layer.

    SiT / REPA:
        Input:  [B, N, D]
        Output: [B, N, P*P*C]

    REG:
        Input:  [B, N+1, D]
        Output: image patch predictions + CLS prediction.
    """

    hidden_size: int
    patch_size: int
    out_channels: int

    predict_cls: bool = False
    cls_token_dim: int = 768

    @nn.compact
    def __call__(self, x, c):

        # c: [B, D]

        # ------------------------------------------------
        # 1. Conditioning modulation
        # ------------------------------------------------

        modulation = nn.Dense(
            2 * self.hidden_size,
            kernel_init=nn.initializers.zeros,
            bias_init=nn.initializers.zeros
        )(nn.silu(c))

        shift, scale = jnp.split(
            modulation, 2, axis=-1
        )

        # ------------------------------------------------
        # 2. Layer normalization + AdaLN
        # ------------------------------------------------

        x = nn.LayerNorm(
            epsilon=1e-6,
            use_scale=False,
            use_bias=False
        )(x)

        x = modulate(x, shift, scale)

        # ------------------------------------------------
        # 3. Image patch prediction
        # ------------------------------------------------

        if self.predict_cls:

            # First token is CLS.
            cls_hidden = x[:, 0]

            # Remaining tokens are image patches.
            patch_hidden = x[:, 1:]

        else:

            patch_hidden = x

        patch_tokens = nn.Dense(
            self.patch_size ** 2 * self.out_channels,
            kernel_init=nn.initializers.zeros,
            bias_init=nn.initializers.zeros
        )(patch_hidden)

        # ------------------------------------------------
        # 4. Optional CLS prediction (REG)
        # ------------------------------------------------

        if self.predict_cls:

            cls_prediction = nn.Dense(
                self.cls_token_dim,
                kernel_init=nn.initializers.zeros,
                bias_init=nn.initializers.zeros
            )(cls_hidden)

        else:

            cls_prediction = None

        return patch_tokens, cls_prediction

## Patch Embeddings

In [9]:
def get_1d_sincos_pos_embed(embed_dim, pos):
    """
    Generates 1D sine-cosine positional embeddings.

    Args:
        embed_dim: D, embedding dimension (must be even).
        pos: [M], M positions.

    Returns:
        [M, D], positional embeddings.
    """

    # Generate D/2 frequency indices: [D/2]
    omega = jnp.arange(embed_dim // 2, dtype=jnp.float32)

    # Normalize frequency indices to [0, 2/D, ..., (D-2)/D]
    omega = omega / (embed_dim / 2.0)

    # Convert indices into exponentially decreasing frequencies.
    # omega[d] = 1 / 10000^(2d/D)
    omega = 1.0 / (10000 ** omega)  # [D/2]

    # Outer product: multiply every position by every frequency.
    # [M, 1] * [1, D/2] -> [M, D/2] (each row contains each postiion multiplied by all frequencies)
    out = jnp.einsum("m,d->md", pos.reshape(-1), omega)

    # Encode each position using sine and cosine.
    # Concatenate along feature dimension: [M, D/2] + [M, D/2]
    # Result: [M, D]
    return jnp.concatenate(
        [jnp.sin(out), jnp.cos(out)],
        axis=-1
    )


def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """
    Generates 2D sine-cosine positional embeddings.

    Args:
        embed_dim: D, total embedding dimension (must be divisible by 4).
        grid_size: G, number of patches along each spatial dimension.

    Returns:
        [G*G, D], positional embeddings for all spatial positions.
    """

    # Generate horizontal and vertical coordinate indices: [G]
    grid_h = jnp.arange(grid_size, dtype=jnp.float32)
    grid_w = jnp.arange(grid_size, dtype=jnp.float32)

    # Construct 2D coordinate grids, each of shape [G, G].
    # grid[0] contains horizontal coordinates (columns).
    # grid[1] contains vertical coordinates (rows).
    grid = jnp.meshgrid(grid_w, grid_h, indexing="xy")

    # Encode horizontal coordinates using D/2 features.
    # Input coordinates: [G, G]
    # Flattened positions: [G*G]
    # Output: [G*G, D/2]
    emb_h = get_1d_sincos_pos_embed(
        embed_dim // 2, grid[0]
    )

    # Encode vertical coordinates using the remaining D/2 features.
    # Output: [G*G, D/2]
    emb_w = get_1d_sincos_pos_embed(
        embed_dim // 2, grid[1]
    )

    # Concatenate horizontal and vertical embeddings.
    # [G*G, D/2] + [G*G, D/2] -> [G*G, D]
    return jnp.concatenate([emb_h, emb_w], axis=-1)


class PatchEmbed(nn.Module):
    """
    Converts a spatial latent representation into patch tokens.
    """

    input_size: int
    patch_size: int
    in_channels: int
    hidden_size: int

    @nn.compact
    def __call__(self, x):
        """
        Args:
            x: [B, H, W, C]

        Returns:
            x: [B, N, D]
        """

        # B = batch size
        # H, W = latent spatial dimensions
        # C = input channels
        # P = patch size
        # D = embedding dimension
        # N = number of patches

        B, H, W, C = x.shape

        P = self.patch_size
        D = self.hidden_size

        assert H == self.input_size and W == self.input_size
        assert C == self.in_channels
        assert H % P == 0 and W % P == 0

        # Project each non-overlapping P x P patch into D dimensions.
        # [B, H, W, C] -> [B, H/P, W/P, D]
        x = nn.Conv(
            features=D,
            kernel_size=(P, P),
            strides=(P, P),
            padding="VALID",
            use_bias=True
        )(x)

        # Flatten the spatial grid into a sequence of N patch tokens.
        # [B, H/P, W/P, D] -> [B, N, D]
        x = x.reshape(B, -1, D)

        return x

## Unified architecture (SiT, REPA, REG)

In [10]:
class SiT(nn.Module):
    """
    Unified SiT architecture supporting:

        variant="sit"  : Standard SiT
        variant="repa" : SiT + DINO representation alignment
        variant="reg"  : Representation Entanglement Generation

    Differences are controlled through the variant parameter.
    """

    # ------------------------------------------------
    # Variant
    # ------------------------------------------------

    variant: str = "sit"

    # ------------------------------------------------
    # Image configuration
    # ------------------------------------------------

    input_size: int = 32
    patch_size: int = 2
    in_channels: int = 4

    # ------------------------------------------------
    # Transformer configuration
    # ------------------------------------------------

    hidden_size: int = 384
    decoder_hidden_size: int = 384

    encoder_depth: int = 8
    depth: int = 12

    num_heads: int = 6
    mlp_ratio: float = 4.0

    # ------------------------------------------------
    # Conditioning
    # ------------------------------------------------

    num_classes: int = 1000
    class_dropout_prob: float = 0.1

    # ------------------------------------------------
    # Representation alignment
    # ------------------------------------------------

    z_dims: tuple = (768,)
    projector_dim: int = 2048

    qk_norm: bool = False

    # =================================================
    # SETUP
    # =================================================

    def setup(self):

        assert self.variant in ("sit", "repa", "reg")

        assert self.hidden_size == self.decoder_hidden_size
        assert 0 < self.encoder_depth <= self.depth

        assert self.input_size % self.patch_size == 0
        assert self.hidden_size % self.num_heads == 0

        grid_size = self.input_size // self.patch_size

        # ------------------------------------------------
        # 1. Patch embedding
        # ------------------------------------------------

        self.x_embedder = PatchEmbed(
            input_size=self.input_size,
            patch_size=self.patch_size,
            in_channels=self.in_channels,
            hidden_size=self.hidden_size
        )

        # ------------------------------------------------
        # 2. Timestep and class embeddings
        # ------------------------------------------------

        self.t_embedder = TimestepEmbedder(
            hidden_size=self.hidden_size
        )

        self.y_embedder = LabelEmbedder(
            num_classes=self.num_classes,
            hidden_size=self.hidden_size,
            dropout_prob=self.class_dropout_prob
        )

        # ------------------------------------------------
        # 3. Positional embeddings
        # ------------------------------------------------

        pos_embed = get_2d_sincos_pos_embed(
            self.hidden_size,
            grid_size
        )

        # REG has an additional CLS token.
        if self.variant == "reg":

            cls_pos_embed = jnp.zeros(
                (1, self.hidden_size)
            )

            pos_embed = jnp.concatenate(
                [cls_pos_embed, pos_embed],
                axis=0
            )

        self.pos_embed = self.variable(
            "constants",
            "pos_embed",
            lambda: pos_embed[None, :, :]
        )

        # ------------------------------------------------
        # 4. Transformer blocks
        # ------------------------------------------------

        self.blocks = [
            SiTBlock(
                hidden_size=self.hidden_size,
                num_heads=self.num_heads,
                mlp_ratio=self.mlp_ratio,
                qk_norm=self.qk_norm
            )
            for _ in range(self.depth)
        ]

        # ------------------------------------------------
        # 5. Intermediate representation projectors
        # ------------------------------------------------

        if self.variant in ("repa", "reg"):

            self.projectors = [
                Projector(
                    projector_dim=self.projector_dim,
                    z_dim=z_dim
                )
                for z_dim in self.z_dims
            ]

        # ------------------------------------------------
        # 6. External CLS input pathway (REG only)
        # ------------------------------------------------

        if self.variant == "reg":

            self.cls_projectors2 = nn.Dense(
                self.hidden_size
            )

            self.wg_norm = nn.LayerNorm(
                epsilon=1e-6
            )

        # ------------------------------------------------
        # 7. Configurable final layer
        # ------------------------------------------------

        self.final_layer = FinalLayer(
            hidden_size=self.hidden_size,
            patch_size=self.patch_size,
            out_channels=self.in_channels,

            predict_cls=(self.variant == "reg"),

            cls_token_dim=self.z_dims[0]
        )

    # =================================================
    # UNPATCHIFY
    # =================================================

    def unpatchify(self, x):

        # x: [B, N, P*P*C]

        B = x.shape[0]

        P = self.patch_size
        C = self.in_channels

        grid_size = self.input_size // P

        x = x.reshape(
            B,
            grid_size,
            grid_size,
            P,
            P,
            C
        )

        x = jnp.transpose(
            x,
            (0, 1, 3, 2, 4, 5)
        )

        x = x.reshape(
            B,
            grid_size * P,
            grid_size * P,
            C
        )

        return x

    # =================================================
    # FORWARD PASS
    # =================================================

    def __call__(
        self,
        x,
        t,
        y,
        cls_token=None,
        train=False,
        force_drop_ids=None,
        debug=False
    ):

        def print_shape(name, tensor):

            if debug and tensor is not None:
                print(f"{name}: {tensor.shape}")

        # ------------------------------------------------
        # 1. Patch embedding
        # ------------------------------------------------

        print_shape("Input latent", x)

        x = self.x_embedder(x)

        print_shape("Patch embeddings", x)

        # ------------------------------------------------
        # 2. Optional external CLS input (REG)
        # ------------------------------------------------

        if self.variant == "reg":

            if cls_token is None:
                raise ValueError(
                    "REG requires an external CLS token."
                )

            cls_token = self.cls_projectors2(
                cls_token
            )

            cls_token = self.wg_norm(
                cls_token
            )

            cls_token = cls_token[:, None, :]

            x = jnp.concatenate(
                [cls_token, x],
                axis=1
            )

            print_shape("Sequence with CLS", x)

        else:

            if cls_token is not None:
                raise ValueError(
                    "External CLS input is only supported by REG."
                )

        # ------------------------------------------------
        # 3. Positional embeddings
        # ------------------------------------------------

        x = x + self.pos_embed.value

        print_shape("Position-encoded sequence", x)

        # ------------------------------------------------
        # 4. Timestep and class conditioning
        # ------------------------------------------------

        t_embed = self.t_embedder(t)

        y_embed = self.y_embedder(
            y,
            train=train,
            force_drop_ids=force_drop_ids
        )

        c = t_embed + y_embed

        print_shape("Conditioning", c)

        # ------------------------------------------------
        # 5. Transformer blocks
        # ------------------------------------------------

        zs = None

        for i, block in enumerate(self.blocks):

            x = block(x, c)

            if debug:
                print(
                    f"Transformer block {i + 1}: {x.shape}"
                )

            # Extract intermediate representation.
            if (i + 1) == self.encoder_depth:

                if self.variant in ("repa", "reg"):

                    zs = [
                        projector(x)
                        for projector in self.projectors
                    ]

                    if debug:

                        for j, z in enumerate(zs):
                            print(
                                f"Projector {j}: {z.shape}"
                            )

        # ------------------------------------------------
        # 6. Final prediction
        # ------------------------------------------------

        patch_tokens, cls_prediction = self.final_layer(
            x,
            c
        )

        print_shape("Patch predictions", patch_tokens)
        print_shape("CLS prediction", cls_prediction)

        # ------------------------------------------------
        # 7. Unpatchify image velocity
        # ------------------------------------------------

        velocity = self.unpatchify(
            patch_tokens
        )

        print_shape("Image velocity", velocity)

        return velocity, zs, cls_prediction

## Test SiT

In [11]:
# ================================================
# PURE SiT
# ================================================

model = SiT(
    variant="sit",

    input_size=32,
    patch_size=2,
    in_channels=4,

    hidden_size=384,
    decoder_hidden_size=384,

    encoder_depth=8,
    depth=12,

    num_heads=6,
    mlp_ratio=4.0,

    num_classes=1000,
    class_dropout_prob=0.1,

    z_dims=(768,),
    projector_dim=2048,

    qk_norm=False
)

B = 2

# Noisy VAE latent
x = jax.random.normal(
    jax.random.PRNGKey(0),
    (B, 32, 32, 4)
)

# Diffusion timestep
t = jnp.array(
    [100, 500],
    dtype=jnp.float32
)

# Class labels
y = jnp.array(
    [10, 25],
    dtype=jnp.int32
)

rng = jax.random.PRNGKey(42)

params_rng, dropout_rng = jax.random.split(rng)

# Initialize
variables = model.init(
    {
        "params": params_rng,
        "dropout": dropout_rng,
    },
    x,
    t,
    y,
    train=True
)

# Forward pass
output, zs, cls_prediction = model.apply(
    variables,
    x,
    t,
    y,
    train=False,
    debug=True
)

print("\n========== PURE SiT RESULTS ==========")

print("Input shape:", x.shape)
print("Output shape:", output.shape)

print("Projectors:", zs)
print("CLS prediction:", cls_prediction)

print("\n========== NUMERICAL CHECKS ==========")

print("Output finite:", jnp.all(jnp.isfinite(output)))
print("Output mean:", jnp.mean(output))
print("Output std:", jnp.std(output))

Input latent: (2, 32, 32, 4)
Patch embeddings: (2, 256, 384)
Position-encoded sequence: (2, 256, 384)
Conditioning: (2, 384)
Transformer block 1: (2, 256, 384)
Transformer block 2: (2, 256, 384)
Transformer block 3: (2, 256, 384)
Transformer block 4: (2, 256, 384)
Transformer block 5: (2, 256, 384)
Transformer block 6: (2, 256, 384)
Transformer block 7: (2, 256, 384)
Transformer block 8: (2, 256, 384)
Transformer block 9: (2, 256, 384)
Transformer block 10: (2, 256, 384)
Transformer block 11: (2, 256, 384)
Transformer block 12: (2, 256, 384)
Patch predictions: (2, 256, 16)
Image velocity: (2, 32, 32, 4)

========== PURE SiT RESULTS ==========
Input shape: (2, 32, 32, 4)
Output shape: (2, 32, 32, 4)
Projectors: None
CLS prediction: None

========== NUMERICAL CHECKS ==========
Output finite: True
Output mean: 0.0
Output std: 0.0


## Test REPA

In [12]:
# ================================================
# REPA
# ================================================

model = SiT(
    variant="repa",

    input_size=32,
    patch_size=2,
    in_channels=4,

    hidden_size=384,
    decoder_hidden_size=384,

    encoder_depth=8,
    depth=12,

    num_heads=6,
    mlp_ratio=4.0,

    num_classes=1000,
    class_dropout_prob=0.1,

    z_dims=(768,),
    projector_dim=2048,

    qk_norm=False
)

B = 2

# Noisy VAE latent
x = jax.random.normal(
    jax.random.PRNGKey(0),
    (B, 32, 32, 4)
)

# Diffusion timestep
t = jnp.array(
    [100, 500],
    dtype=jnp.float32
)

# Class labels
y = jnp.array(
    [10, 25],
    dtype=jnp.int32
)

rng = jax.random.PRNGKey(42)

params_rng, dropout_rng = jax.random.split(rng)

# Initialize
variables = model.init(
    {
        "params": params_rng,
        "dropout": dropout_rng,
    },
    x,
    t,
    y,
    train=True
)

# Forward pass
output, zs, cls_prediction = model.apply(
    variables,
    x,
    t,
    y,
    train=False,
    debug=True
)

print("\n========== REPA RESULTS ==========")

print("Input shape:", x.shape)
print("Output shape:", output.shape)

print("Number of projectors:", len(zs))

for i, z in enumerate(zs):
    print(f"Projector {i} output shape:", z.shape)

print("CLS prediction:", cls_prediction)

print("\n========== NUMERICAL CHECKS ==========")

print("Output finite:", jnp.all(jnp.isfinite(output)))

for i, z in enumerate(zs):
    print(
        f"Projector {i} finite:",
        jnp.all(jnp.isfinite(z))
    )

print("Output mean:", jnp.mean(output))
print("Output std:", jnp.std(output))

Input latent: (2, 32, 32, 4)
Patch embeddings: (2, 256, 384)
Position-encoded sequence: (2, 256, 384)
Conditioning: (2, 384)
Transformer block 1: (2, 256, 384)
Transformer block 2: (2, 256, 384)
Transformer block 3: (2, 256, 384)
Transformer block 4: (2, 256, 384)
Transformer block 5: (2, 256, 384)
Transformer block 6: (2, 256, 384)
Transformer block 7: (2, 256, 384)
Transformer block 8: (2, 256, 384)
Projector 0: (2, 256, 768)
Transformer block 9: (2, 256, 384)
Transformer block 10: (2, 256, 384)
Transformer block 11: (2, 256, 384)
Transformer block 12: (2, 256, 384)
Patch predictions: (2, 256, 16)
Image velocity: (2, 32, 32, 4)

========== REPA RESULTS ==========
Input shape: (2, 32, 32, 4)
Output shape: (2, 32, 32, 4)
Number of projectors: 1
Projector 0 output shape: (2, 256, 768)
CLS prediction: None

========== NUMERICAL CHECKS ==========
Output finite: True
Projector 0 finite: True
Output mean: 0.0
Output std: 0.0


## Test REG

In [13]:
# ================================================
# REG
# ================================================

model = SiT(
    variant="reg",

    input_size=32,
    patch_size=2,
    in_channels=4,

    hidden_size=384,
    decoder_hidden_size=384,

    encoder_depth=8,
    depth=12,

    num_heads=6,
    mlp_ratio=4.0,

    num_classes=1000,
    class_dropout_prob=0.1,

    z_dims=(768,),
    projector_dim=2048,

    qk_norm=False
)

B = 2

# Noisy VAE latent
x = jax.random.normal(
    jax.random.PRNGKey(0),
    (B, 32, 32, 4)
)

# Diffusion timestep
t = jnp.array(
    [100, 500],
    dtype=jnp.float32
)

# Class labels
y = jnp.array(
    [10, 25],
    dtype=jnp.int32
)

# External DINO CLS representation
cls_token = jax.random.normal(
    jax.random.PRNGKey(1),
    (B, 768)
)

rng = jax.random.PRNGKey(42)

params_rng, dropout_rng = jax.random.split(rng)

# Initialize
variables = model.init(
    {
        "params": params_rng,
        "dropout": dropout_rng,
    },
    x,
    t,
    y,
    cls_token,
    train=True
)

# Forward pass
output, zs, cls_prediction = model.apply(
    variables,
    x,
    t,
    y,
    cls_token,
    train=False,
    debug=True
)

print("\n========== REG RESULTS ==========")

print("Input shape:", x.shape)
print("External CLS shape:", cls_token.shape)

print("Output shape:", output.shape)

print("Number of projectors:", len(zs))

for i, z in enumerate(zs):
    print(f"Projector {i} output shape:", z.shape)

print("CLS prediction shape:", cls_prediction.shape)

print("\n========== NUMERICAL CHECKS ==========")

print("Output finite:", jnp.all(jnp.isfinite(output)))

for i, z in enumerate(zs):
    print(
        f"Projector {i} finite:",
        jnp.all(jnp.isfinite(z))
    )

print(
    "CLS prediction finite:",
    jnp.all(jnp.isfinite(cls_prediction))
)

print("Output mean:", jnp.mean(output))
print("Output std:", jnp.std(output))

Input latent: (2, 32, 32, 4)
Patch embeddings: (2, 256, 384)
Sequence with CLS: (2, 257, 384)
Position-encoded sequence: (2, 257, 384)
Conditioning: (2, 384)
Transformer block 1: (2, 257, 384)
Transformer block 2: (2, 257, 384)
Transformer block 3: (2, 257, 384)
Transformer block 4: (2, 257, 384)
Transformer block 5: (2, 257, 384)
Transformer block 6: (2, 257, 384)
Transformer block 7: (2, 257, 384)
Transformer block 8: (2, 257, 384)
Projector 0: (2, 257, 768)
Transformer block 9: (2, 257, 384)
Transformer block 10: (2, 257, 384)
Transformer block 11: (2, 257, 384)
Transformer block 12: (2, 257, 384)
Patch predictions: (2, 256, 16)
CLS prediction: (2, 768)
Image velocity: (2, 32, 32, 4)

========== REG RESULTS ==========
Input shape: (2, 32, 32, 4)
External CLS shape: (2, 768)
Output shape: (2, 32, 32, 4)
Number of projectors: 1
Projector 0 output shape: (2, 257, 768)
CLS prediction shape: (2, 768)

========== NUMERICAL CHECKS ==========
Output finite: True
Projector 0 finite: True
CLS